# 🧠 Sentiment Analysis Using CNN
**Binary Sentiment Classification on IMDB Movie Reviews**

**Tech Stack:** Python · Keras · NLP (NLTK) · scikit-learn  
**Author:** Singamsetti Keerthi  

---

## Objective
Build a 1D Convolutional Neural Network (CNN) to classify movie reviews as **Positive** or **Negative**.

## Pipeline
```
Raw Text → Preprocessing → Tokenization → Padding → CNN → Binary Output
```

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, pickle

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords')

from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

# ── Hyperparameters ──────────────────────────────────────────
VOCAB_SIZE    = 10_000
MAX_LEN       = 200
EMBEDDING_DIM = 64
NUM_FILTERS   = 128
KERNEL_SIZE   = 5
BATCH_SIZE    = 64
EPOCHS        = 15

print('✅ Imports done')

## 2. Load & Explore Dataset

In [ ]:
# Load IMDB dataset (built into Keras)
(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f'Training samples : {len(X_train_raw):,}')
print(f'Test samples     : {len(X_test_raw):,}')
print(f'Positive (train) : {y_train.sum():,}')
print(f'Negative (train) : {(1-y_train).sum():,}')
print(f'\nSample review (encoded): {X_train_raw[0][:20]} …')
print(f'Sample label           : {"Positive" if y_train[0]==1 else "Negative"}')

In [ ]:
# Review length distribution
lengths = [len(r) for r in X_train_raw]
plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, color='#6366f1', alpha=0.8, edgecolor='white')
plt.axvline(MAX_LEN, color='red', linestyle='--', label=f'MAX_LEN={MAX_LEN}')
plt.title('Distribution of Review Lengths', fontsize=13, fontweight='bold')
plt.xlabel('Number of Words'); plt.ylabel('Count')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Mean length  : {np.mean(lengths):.0f}')
print(f'Median length: {np.median(lengths):.0f}')
print(f'Max length   : {np.max(lengths)}')

## 3. Text Preprocessing

In [ ]:
# Decode a review to see actual text
word_index = imdb.get_word_index()
index_word = {v+3: k for k, v in word_index.items()}
index_word.update({0: '<PAD>', 1: '<START>', 2: '<UNK>', 3: '<UNUSED>'})

def decode_review(encoded):
    return ' '.join(index_word.get(i, '?') for i in encoded)

print('📄 Sample review (decoded):')
print(decode_review(X_train_raw[0])[:400], '…')
print(f'\nLabel: {"Positive ✅" if y_train[0]==1 else "Negative ❌"}')

In [ ]:
# Pad / truncate all sequences to MAX_LEN
X_train = pad_sequences(X_train_raw, maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(X_test_raw,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')

## 4. Build CNN Model

In [ ]:
model = Sequential([
    # Word embeddings — learnt during training
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN, name='embedding'),

    # Conv block 1 — captures local n-gram patterns
    Conv1D(NUM_FILTERS, KERNEL_SIZE, activation='relu', padding='same', name='conv1'),
    BatchNormalization(name='bn1'),
    Dropout(0.4, name='drop1'),

    # Conv block 2 — deeper feature extraction
    Conv1D(NUM_FILTERS//2, 3, activation='relu', padding='same', name='conv2'),
    BatchNormalization(name='bn2'),

    # Global max pool — most important feature from entire sequence
    GlobalMaxPooling1D(name='global_pool'),

    # Classification head
    Dense(64, activation='relu', name='dense1'),
    Dropout(0.4, name='drop2'),
    Dense(1, activation='sigmoid', name='output'),   # binary output
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 5. Train the Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN Sentiment Analysis — Training History', fontsize=14, fontweight='bold')

ax1.plot(history.history['accuracy'],     label='Train', lw=2)
ax1.plot(history.history['val_accuracy'], label='Validation', lw=2, ls='--')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train', lw=2)
ax2.plot(history.history['val_loss'], label='Validation', lw=2, ls='--')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluation

In [ ]:
y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)

print(f'Test Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'ROC-AUC Score : {roc_auc_score(y_test, y_pred_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout(); plt.show()

## 8. Predict on New Reviews

In [ ]:
stemmer    = PorterStemmer()
stop_words = set(stopwords.words('english')) - {'not', 'no', 'never'}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return ' '.join(stemmer.stem(w) for w in text.split()
                    if w not in stop_words and len(w) > 1)

def predict_review(text):
    cleaned = preprocess(text)
    seq = [word_index.get(w, 2)+3 for w in cleaned.split() if word_index.get(w,2)+3 < VOCAB_SIZE]
    padded = pad_sequences([seq], maxlen=MAX_LEN, padding='post', truncating='post')
    prob   = float(model.predict(padded, verbose=0)[0][0])
    label  = '✅ POSITIVE' if prob >= 0.5 else '❌ NEGATIVE'
    conf   = prob if prob >= 0.5 else 1-prob
    print(f'  {label}  |  Confidence: {conf*100:.1f}%  |  Score: {prob:.4f}')

reviews = [
    "This movie was absolutely brilliant! The acting was superb and the story was gripping.",
    "Terrible film. Boring plot, horrible acting and a complete waste of time.",
    "One of the best movies I have ever seen. Highly recommend!",
    "Not bad, but could have been much better with a stronger script.",
]

print('🔮 Predictions on Custom Reviews\n' + '='*50)
for r in reviews:
    print(f'\n  Review: {r[:70]}…')
    predict_review(r)

## 9. Save the Model

In [ ]:
import os
os.makedirs('model', exist_ok=True)

model.save('model/sentiment_cnn_final.keras')
with open('model/word_index.pkl', 'wb') as f:
    pickle.dump(word_index, f)

print('✅ Model saved to model/sentiment_cnn_final.keras')
print('✅ Word index saved to model/word_index.pkl')